# SE(3) invariance demo

DHB invariants are invariant to global SE(3) transforms: if we apply the same rigid transform to a trajectory, the invariant sequence (magnitudes and relative frame rotations) is unchanged.

In [ ]:
import numpy as np
from dhb_xr.encoder.dhb_dr import encode_dhb_dr
from dhb_xr.core.types import DHBMethod
from dhb_xr.core import geometry as geom

def apply_se3(positions, quaternions, R, t):
    p_new = (R @ positions.T).T + t
    q_new = np.array([geom.rot_to_quat(R @ geom.quat_to_rot(q)) for q in quaternions])
    return p_new, q_new

n = 40
np.random.seed(123)
positions = np.cumsum(np.random.randn(n, 3) * 0.02, axis=0)
quats = np.tile(np.array([1.0, 0.0, 0.0, 0.0]), (n, 1))
out1 = encode_dhb_dr(positions, quats, method="pos", use_default_initial_frames=True, dhb_method=DHBMethod.DOUBLE_REFLECTION)

axis = np.random.randn(3); axis /= np.linalg.norm(axis)
R = geom.axis_angle_to_rot(axis * 0.5)
t = np.array([0.1, -0.2, 0.05])
p2, q2 = apply_se3(positions, quats, R, t)
out2 = encode_dhb_dr(p2, q2, method="pos", use_default_initial_frames=True, dhb_method=DHBMethod.DOUBLE_REFLECTION)

print("Magnitude invariance (first column):")
print("  Linear  match:", np.allclose(out1["linear_motion_invariants"][:, 0], out2["linear_motion_invariants"][:, 0]))
print("  Angular match:", np.allclose(out1["angular_motion_invariants"][:, 0], out2["angular_motion_invariants"][:, 0]))